In [16]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


In [23]:
X = pd.read_parquet('../data/output/features.parquet')
y = X['target']
X = X.drop(columns='target')

groups = X.index.get_level_values('id')
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

display(X_train.head())
display(y_train.head())

Training set shape: (4018147, 8)
Testing set shape: (1018370, 8)


mean_diff  median_diff  std_diff  skew_diff  kurt_diff  min_diff  \
id time                                                                     
1  1339  -0.564726    -0.567470       NaN        NaN        NaN  2.783253   
   1340  -0.588736    -0.591480 -0.966419        NaN        NaN  2.735234   
   1341  -0.474306    -0.567470 -0.800727   1.596376        NaN  2.735234   
   1342  -0.303484    -0.407831 -0.621833   1.050229   0.195134  2.735234   
   1343  -0.171257    -0.248192 -0.558910   0.245438  -2.441037  2.735234   

         max_diff  autocorr_diff  
id time                           
1  1339 -4.036022            NaN  
   1340 -4.036022            NaN  
   1341 -3.716744      -1.422394  
   1342 -3.262314       0.412416  
   1343 -3.113645       0.437092

id  time
1   1339    0
    1340    0
    1341    0
    1342    0
    1343    0
Name: target, dtype: int64

In [18]:
start = time.time()
clf = HistGradientBoostingClassifier()
clf.fit(X_train, y_train)

print("Training time: {:.2f} seconds".format(time.time() - start))

Training time: 5.77 seconds


In [19]:
y_pred_proba = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print("Baseline HistGradientBoosting has a ROC AUC:", auc)

Baseline HistGradientBoosting has a ROC AUC: 0.6530853219230861


In [20]:
# Finetune the model using GridSearchCV
print("List of Params: ", clf.get_params())

start = time.time()

param_grids = {
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "max_iter": [100, 200, 300],
    "max_leaf_nodes": [31, 63, 127],
    "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 0.1, 0.5],
    "class_weight": [None, "balanced"]
}

group_kfold = GroupKFold(n_splits=5, shuffle=True, random_state=42)

grid = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(),
    param_distributions=param_grids,
    scoring='roc_auc',
    cv=group_kfold,
    n_jobs=-1,
    verbose=1,
    n_iter=50
)

groups_train = X_train.index.get_level_values('id')

grid.fit(X_train, y_train, groups=groups_train)

print("Finetuning time: {:.2f} seconds".format(time.time() - start))
print("Best Params: ", grid.best_params_)
print("Best Score: ", grid.best_score_)

List of Params:  {'categorical_features': 'from_dtype', 'class_weight': None, 'early_stopping': 'auto', 'interaction_cst': None, 'l2_regularization': 0.0, 'learning_rate': 0.1, 'loss': 'log_loss', 'max_bins': 255, 'max_depth': None, 'max_features': 1.0, 'max_iter': 100, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'monotonic_cst': None, 'n_iter_no_change': 10, 'random_state': None, 'scoring': 'loss', 'tol': 1e-07, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Finetuning time: 1241.86 seconds
Best Params:  {'min_samples_leaf': 100, 'max_leaf_nodes': 127, 'max_iter': 200, 'max_depth': 3, 'learning_rate': 0.1, 'l2_regularization': 0.0, 'class_weight': 'balanced'}
Best Score:  0.6307479957375053


In [21]:
best_clf = grid.best_estimator_
y_pred_proba_tuned = best_clf.predict_proba(X_test)[:, 1]
auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)
print("Tuned model ROC AUC on X_test:", auc_tuned)

Tuned model ROC AUC on X_test: 0.6535398705129062


mean_diff  median_diff  std_diff  skew_diff  kurt_diff  min_diff  \
id time                                                                     
1  1339  -0.564726    -0.567470       NaN        NaN        NaN  2.783253   
   1340  -0.588736    -0.591480 -0.966419        NaN        NaN  2.735234   
   1341  -0.474306    -0.567470 -0.800727   1.596376        NaN  2.735234   
   1342  -0.303484    -0.407831 -0.621833   1.050229   0.195134  2.735234   
   1343  -0.171257    -0.248192 -0.558910   0.245438  -2.441037  2.735234   

         max_diff  autocorr_diff  
id time                           
1  1339 -4.036022            NaN  
   1340 -4.036022            NaN  
   1341 -3.716744      -1.422394  
   1342 -3.262314       0.412416  
   1343 -3.113645       0.437092

id  time
1   1339    0
    1340    0
    1341    0
    1342    0
    1343    0
Name: target, dtype: int64

# Data Split Visualized

![Train/Validated/Test Split](../notes/images/data_split_visualized.svg)